####Segment_Rules

In [0]:
from pyspark.sql.window import Window

In [0]:
#Load the RFM table
from pyspark.sql import functions as F

rfm_df = spark.table("customer_rfm")
customer_df = spark.table("customer_master")

display(rfm_df)

In [0]:
#Create the SegmentRule table
segment_rules = [
    ("SEG-2026-000001", "Champions", "Recent customers with high frequency and spending", 1),
    ("SEG-2026-000002", "Loyal", "Customers with good purchase frequency", 2),
    ("SEG-2026-000003", "New Customers", "Recently registered customers with few purchases", 3),
    ("SEG-2026-000004", "At Risk", "Previously active customers with low recency score", 4),
    ("SEG-2026-000005", "Hibernating", "Customers with no recent purchases and low frequency", 5),
    ("SEG-2026-000006", "Developing", "Customers who do not match other segment rules", 6)
]

segment_rule_df = spark.createDataFrame(
    segment_rules,
    ["segment_id", "segment_name", "description", "priority"]
)

segment_rule_df = (
    segment_rule_df
    .withColumn("effective_from", F.lit("2026-10-30").cast("date"))
    .withColumn("effective_to", F.lit(None).cast("date"))
    .withColumn("rule_status", F.lit("ACTIVE"))
)

display(segment_rule_df)

In [0]:
#Save the SegmentRule table (notebook 12 validates segment_membership against it)
segment_rule_df.write.format("delta").mode("overwrite").saveAsTable("segment_rule")

In [0]:
#Calculate RFM scores
purchase_df = rfm_df.filter(F.col("frequency") > 0)

from pyspark.sql.window import Window

# Recency is "days since last purchase", so a LOWER value is better.
# Ordering by recency DESC puts the oldest purchasers in bucket 1 and the most
# recent purchasers in bucket 5, so ntile(5) is already the score (5 = best).
recency_window = Window.orderBy(F.col("recency").desc())
frequency_window = Window.orderBy(F.col("frequency").asc())
monetary_window = Window.orderBy(F.col("monetary").asc())

scored_purchase_df = (
    purchase_df
    .withColumn("r_score", F.ntile(5).over(recency_window))
    .withColumn("f_score", F.ntile(5).over(frequency_window))
    .withColumn("m_score", F.ntile(5).over(monetary_window))
)

scored_purchase_df = scored_purchase_df.withColumn(
    "rfm_score",
    F.col("r_score") + F.col("f_score") + F.col("m_score")
)

display(scored_purchase_df)

In [0]:
no_purchase_df = (
    rfm_df
    .filter(F.col("frequency") == 0)
    .withColumn("r_score", F.lit(0))
    .withColumn("f_score", F.lit(0))
    .withColumn("m_score", F.lit(0))
    .withColumn("rfm_score", F.lit(0))
)

In [0]:
#Combine all customer scores
all_scored_df = scored_purchase_df.unionByName(no_purchase_df)

display(all_scored_df)

In [0]:
#Assign customer segments
segmented_df = (
    all_scored_df
    .withColumn(
        "segment_name",
        F.when(
            (F.col("r_score") >= 4) &
            (F.col("f_score") >= 4) &
            (F.col("m_score") >= 4),
            "Champions"
        )
        .when(
            (F.col("f_score") >= 4) &
            (F.col("r_score") >= 3),
            "Loyal"
        )
        .when(
            (F.col("frequency") <= 1) &
            (F.col("signup_date") >= F.date_sub(F.lit("2026-10-30").cast("date"), 30)),
            "New Customers"
        )
        .when(
            (F.col("frequency") >= 1) &
            (F.col("r_score") <= 2) &
            (F.col("f_score") >= 3),
            "At Risk"
        )
        .when(
            (F.col("r_score") <= 2) &
            (F.col("f_score") <= 2),
            "Hibernating"
        )
        .otherwise("Developing")
    )
)

display(segmented_df.groupBy("segment_name").count())

In [0]:
#Create SegmentMembership
segment_lookup = segment_rule_df.select(
    "segment_id",
    "segment_name"
)

membership_df = (
    segmented_df
    .join(segment_lookup, "segment_name", "inner")
    .withColumn(
        "membership_id",
        F.concat(
            F.lit("MEM-"),
            F.col("segment_id"),
            F.lit("-"),
            F.col("customer_id")
        )
    )
    .withColumn("effective_from", F.lit("2026-10-30").cast("date"))
    .withColumn("effective_to", F.lit(None).cast("date"))
    .withColumn("membership_status", F.lit("ACTIVE"))
    .withColumn(
        "reason_text",
        F.concat(
            F.lit("Assigned to "),
            F.col("segment_name"),
            F.lit(" based on RFM scores.")
        )
    )
    .select(
        "membership_id",
        "customer_id",
        "segment_id",
        "segment_name",
        "r_score",
        "f_score",
        "m_score",
        "rfm_score",
        "effective_from",
        "effective_to",
        "membership_status",
        "reason_text"
    )
)

display(membership_df)

In [0]:
membership_df.write.format("delta").mode("overwrite").saveAsTable("segment_membership")

print("Segment rules and customer memberships created successfully!")

In [0]:
#Validate the segmentation
membership_df.groupBy("segment_name").count().orderBy("segment_name").show()

In [0]:
membership_df.groupBy("customer_id").count().filter(
    F.col("count") != 1
).show()